In [44]:
import glob
import math
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import serial
from scipy.interpolate import griddata

# Auto-detect the active Arduino serial port
ports = glob.glob("/dev/ttyACM*") + glob.glob("/dev/ttyUSB*")
if not ports:
    raise RuntimeError(
        "No Arduino found! Check USB cable connection and run 'ls /dev/ttyACM*'"
    )

PORT = ports[0]
BAUD = 115200

print(f"Using serial port: {PORT} at {BAUD} baud")

Using serial port: /dev/ttyACM0 at 115200 baud


In [46]:
# Close any lingering serial connections
try:
  ser.close()
except NameError:
  pass

print(f"Connecting to {PORT}...")
ser = serial.Serial(PORT, BAUD, timeout=1.0)
time.sleep(2.0)  # Allow Arduino DTR auto-reset to settle
ser.reset_input_buffer()

rows = []
scanning = False
start_time = None

print("Listening for Arduino stream... Press Arduino RESET button if idle.")

try:
  while True:
    line = ser.readline().decode(errors="ignore").strip()
    if not line:
      continue

    # Diagnostic & status messages from Arduino
    if line.startswith("#"):
      print("ARDUINO:", line)
      continue

    if line == "START":
      print("\n--- Scan Started ---")
      scanning = True
      start_time = time.time()
      rows.clear()
      continue

    if line == "DONE":
      elapsed = time.time() - start_time if start_time else 0
      print(
          f"\n--- Scan Complete in {elapsed:.1f}s | Captured {len(rows)}"
          " points ---"
      )
      break

    # Parse data: pan,tilt,distance_cm
    parts = line.split(",")
    if len(parts) == 3:
      try:
        pan = float(parts[0])
        tilt = float(parts[1])
        dist = float(parts[2])

        rows.append({"pan": pan, "tilt": tilt, "dist": dist})

        if len(rows) % 100 == 0:
          print(f"Recorded {len(rows)} points...", end="\r")
      except ValueError:
        pass

finally:
  ser.close()
  print("\nSerial connection closed safely.")

# Save captured data
if rows:
  df = pd.DataFrame(rows)
  timestamp = time.strftime("%Y%m%d_%H%M%S")
  filename = f"scan_{timestamp}.csv"
  df.to_csv(filename, index=False)
  df.to_csv("latest_scan.csv", index=False)
  print(f"Data saved to {filename} and latest_scan.csv")
else:
  print("No data points were captured.")

Connecting to /dev/ttyACM0...
Listening for Arduino stream... Press Arduino RESET button if idle.
ARDUINO: # 3D Scanner Ready (Discrete Mode + Torque Kick Fix)

--- Scan Started ---
Recorded 4300 points...
--- Scan Complete in 472.6s | Captured 4331 points ---

Serial connection closed safely.
Data saved to scan_20261114_172728.csv and latest_scan.csv


In [47]:
df = pd.read_csv("latest_scan.csv")

# Filter out out-of-range sensor readings (-1.0)
valid = df[(df["dist"] > 3.0) & (df["dist"] < 200.0)].copy()

if valid.empty:
  print("No valid range measurements found to render.")
else:
  # Convert spherical coordinates to Cartesian coordinates (cm)
  # Pan = horizontal azimuth, Tilt = elevation from vertical
  rad_pan = np.radians(valid["pan"])
  rad_tilt = np.radians(valid["tilt"])

  valid["x"] = valid["dist"] * np.sin(rad_tilt) * np.cos(rad_pan)
  valid["y"] = valid["dist"] * np.sin(rad_tilt) * np.sin(rad_pan)
  valid["z"] = valid["dist"] * np.cos(rad_tilt)

  scatter_fig = go.Figure(
      data=[
          go.Scatter3d(
              x=valid["x"],
              y=valid["y"],
              z=valid["z"],
              mode="markers",
              marker=dict(
                  size=2.5,
                  color=valid["y"],  # Color mapped to depth
                  colorscale="Viridis",
                  colorbar=dict(title="Depth Y (cm)"),
                  showscale=True,
              ),
          )
      ]
  )

  scatter_fig.update_layout(
      template="plotly_dark",
      height=720,
      title=f"3D Point Cloud ({len(valid)} valid points)",
      scene=dict(
          xaxis_title="X, Right (cm)",
          yaxis_title="Y, Depth (cm)",
          zaxis_title="Z, Up (cm)",
          aspectmode="data",
      ),
  )

  # Use show() directly; fallback to browser if nbformat is missing
  try:
    scatter_fig.show()
  except Exception:
    scatter_fig.show(renderer="browser")

In [8]:
if "valid" in locals() and len(valid) >= 16:
  # Create a regular grid across the measured spatial boundary
  grid_x, grid_z = np.mgrid[
      valid["x"].min() : valid["x"].max() : 120j,
      valid["z"].min() : valid["z"].max() : 120j,
  ]

  # Interpolate depth (Y) across the X-Z plane
  points = np.column_stack((valid["x"], valid["z"]))
  grid_y = griddata(points, valid["y"], (grid_x, grid_z), method="cubic")

  # Fallback to linear interpolation for perimeter gaps
  grid_y_linear = griddata(points, valid["y"], (grid_x, grid_z), method="linear")
  grid_y[np.isnan(grid_y)] = grid_y_linear[np.isnan(grid_y)]

  surface_fig = go.Figure(
      data=[
          go.Surface(
              x=grid_x,
              y=grid_y,
              z=grid_z,
              colorscale="Viridis",
              colorbar=dict(title="Depth Y (cm)"),
          )
      ]
  )

  surface_fig.update_layout(
      template="plotly_dark",
      height=750,
      title="3D Topographic Surface Reconstruction",
      scene=dict(
          xaxis_title="X, Right (cm)",
          yaxis_title="Y, Depth (cm)",
          zaxis_title="Z, Up (cm)",
          aspectmode="data",
      ),
  )

  try:
    surface_fig.show()
  except Exception:
    surface_fig.show(renderer="browser")
else:
  print("Insufficient valid points to generate surface interpolation.")

In [11]:
import glob
import os
import webbrowser
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import griddata

# 1. Select which scan file to view
TARGET_CSV = "scan_20261111_175459.csv"  # Set to a specific filename or leave as None for newest scan

if TARGET_CSV is None:
    if os.path.exists("latest_scan.csv"):
        TARGET_CSV = "latest_scan.csv"
    else:
        csv_files = sorted(glob.glob("scan_*.csv"))
        if not csv_files:
            raise FileNotFoundError("No scan CSV files found in the current directory.")
        TARGET_CSV = csv_files[-1]

print(f"Loading scan data from: {TARGET_CSV}")
df = pd.read_csv(TARGET_CSV)

# Normalize distance column name ('dist' or 'dist_cm')
dist_col = "dist" if "dist" in df.columns else "dist_cm"

# 2. Filter valid distance readings (cm)
valid = df[(df[dist_col] > 3.0) & (df[dist_col] < 150.0)].copy()
if valid.empty:
    raise ValueError("No valid distance points found in the selected CSV file.")

# 3. Orientation & Spherical-to-Cartesian projection
# TILT_SIGN = -1 flips the vertical Z-axis so the plot is right-side up
PAN_SIGN  = 1   # Change to -1 if left/right appears mirrored
TILT_SIGN = -1  # Flipped to fix upside-down orientation

az = np.radians(valid["pan"] - 90.0) * PAN_SIGN
el = np.radians(valid["tilt"] - 90.0) * TILT_SIGN
r = valid[dist_col]

valid["x"] = r * np.sin(az) * np.cos(el)  # Left / Right (cm)
valid["y"] = r * np.cos(az) * np.cos(el)  # Forward Depth (cm)
valid["z"] = r * np.sin(el)               # Up / Down (cm)

# 4. Build Side-by-Side Interactive 3D Viewer (Point Cloud + Surface Mesh)
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("3D Point Cloud", "Interpolated 3D Surface"),
    horizontal_spacing=0.03
)

# Left subplot: Raw 3D Scatter Points
fig.add_trace(
    go.Scatter3d(
        x=valid["x"], y=valid["y"], z=valid["z"],
        mode="markers",
        marker=dict(
            size=3,
            color=valid["y"],
            colorscale="Turbo",
            colorbar=dict(title="Depth Y (cm)", x=0.46, thickness=14),
            opacity=0.9
        ),
        name="Points",
        hovertemplate="X: %{x:.1f} cm<br>Y: %{y:.1f} cm<br>Z: %{z:.1f} cm<extra></extra>"
    ),
    row=1, col=1
)

# Right subplot: Continuous 3D Surface
if len(valid) >= 16:
    grid_x, grid_z = np.mgrid[
        valid["x"].min():valid["x"].max():120j,
        valid["z"].min():valid["z"].max():120j
    ]
    pts = np.column_stack((valid["x"], valid["z"]))
    grid_y = griddata(pts, valid["y"], (grid_x, grid_z), method="cubic")
    grid_y_lin = griddata(pts, valid["y"], (grid_x, grid_z), method="linear")
    grid_y[np.isnan(grid_y)] = grid_y_lin[np.isnan(grid_y)]

    fig.add_trace(
        go.Surface(
            x=grid_x, y=grid_y, z=grid_z,
            colorscale="Turbo",
            showscale=False,
            name="Surface",
            hovertemplate="X: %{x:.1f} cm<br>Y: %{y:.1f} cm<br>Z: %{z:.1f} cm<extra></extra>"
        ),
        row=1, col=2
    )

scene_layout = dict(
    xaxis_title="X, Right (cm)",
    yaxis_title="Y, Depth (cm)",
    zaxis_title="Z, Up (cm)",
    aspectmode="data"
)

fig.update_layout(
    template="plotly_dark",
    height=750,
    title=f"3D Scan Viewer — {TARGET_CSV} ({len(valid)} valid points)",
    scene=scene_layout,
    scene2=scene_layout,
    margin=dict(l=10, r=10, t=60, b=10)
)

# 5. Export as standalone HTML and open in browser
html_filename = os.path.splitext(TARGET_CSV)[0] + "_viewer.html"
fig.write_html(html_filename, include_plotlyjs="cdn")

abs_path = os.path.abspath(html_filename)
print(f"Saved interactive HTML viewer to: {abs_path}")
webbrowser.open(f"file://{abs_path}")

Loading scan data from: scan_20261111_175459.csv
Saved interactive HTML viewer to: /home/dilipan/Documents/Projects/Electronics lab/scan_20261111_175459_viewer.html


True